# Workshop: Transfer Learning for Face Image Classification

Welcome! In this workshop you will train and inspect a simple image classifier built on top of a pretrained convolutional neural network.

## By the end of the workshop, you should be able to:
- load an image dataset with PyTorch
- explain the difference between training, validation, and benchmark data
- fine-tune a pretrained model for a new classification task
- evaluate a classifier with more than one metric
- inspect confident mistakes and discuss possible failure modes
- use Grad-CAM to visualize where the model is focusing

> **Important note about the labels**
> This dataset contains two face-image classes provided by the dataset creators. These labels are a simplification of a much more complex real-world concept. In this workshop, we treat the task as a **binary image classification exercise**, not as a reliable system for inferring gender identity from appearance.


## Workshop roadmap

### Part A - Core lab
1. Set up the environment  
2. Load and inspect the data  
3. Build dataloaders  
4. Train one pretrained model (**MobileNetV2**)  
5. Evaluate it on validation and benchmark data  
6. Inspect predictions, mistakes, and Grad-CAM explanations

### Part B - Optional extension at the end
1. Run a few hyperparameter experiments  
2. Save results to TensorBoard  
3. Compare runs visually

Throughout the notebook, look for:
- **Checkpoint questions**: pause and discuss before moving on
- **Your turn** prompts: small things to try or interpret
- **Reflection prompts**: connect the results back to ML practice


## 0. Setup

Run the next cell once. If your environment already has the required packages, it should finish quickly.


In [ ]:
# Optional installs for workshop environments
%pip -q install gitpython grad-cam ipywidgets tensorboard


In [ ]:
from pathlib import Path
import random
import shutil
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from torchvision.models import (
    mobilenet_v2, MobileNet_V2_Weights,
    resnet50, ResNet50_Weights,
    vit_b_16, ViT_B_16_Weights,
)
from torch.utils.tensorboard import SummaryWriter

warnings.filterwarnings("ignore")

try:
    import google.colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

## 1. Workshop control panel

Edit this cell only if you want to change the **baseline run** for the whole notebook.

For the first pass through the workshop, it is usually best to keep the defaults and only experiment later.


In [ ]:
# =========================
# Workshop control panel
# =========================
IMG_SIZE = 224       # use 224 when MODEL_NAME = "vit_b_16"
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.2

EPOCHS = 2
LEARNING_RATE = 1e-4
DROPOUT = 0.2

FREEZE_BACKBONE = True
UNFREEZE_LAST_BLOCK = False   # optional exploration
USE_DATA_AUGMENTATION = True

# Choose your backbone:  "mobilenet_v2"  |  "resnet50"  |  "vit_b_16"
MODEL_NAME = "vit_b_16"

NUM_WORKERS = 0               # set >0 locally if desired

# For quick workshop reruns
SHOW_DATASET_EXAMPLES = 12
RANDOM_EXAMPLE_INDEX = None   # set a fixed int to always show the same example

### Checkpoint question
Before you run the notebook, which settings above do you expect to have the largest effect on:
1. training speed  
2. overfitting  
3. final validation performance

Write down one prediction now. We will come back to it later.


## 2. Get the data

This notebook expects the same dataset structure as the original workshop:

- `faces/train/female`
- `faces/train/male`
- `faces/benchmark/female`
- `faces/benchmark/male`

By default, the next cell clones the public repository used in the original notebook.

### Think before you run
Why might we want a **benchmark** split that stays untouched until the end?


In [ ]:

from git import Repo

if IN_COLAB:
    DATA_PATH = Path("/content/drive/MyDrive/cvlab_workshop")
else:
    DATA_PATH = Path.cwd() / 'data'

DATA_PATH.mkdir(parents=True, exist_ok=True)

FACES_PATH = DATA_PATH / "faces"
TRAIN_PATH = FACES_PATH / "train"
BENCHMARK_PATH = FACES_PATH / "benchmark"

if not FACES_PATH.exists():
    print("Cloning dataset...")
    Repo.clone_from("https://github.com/susuter/faces_red.git", FACES_PATH)

    # Original repo layout: female/, male/, benchmark/
    female_path = FACES_PATH / "female"
    male_path = FACES_PATH / "male"
    TRAIN_PATH.mkdir(parents=True, exist_ok=True)

    if female_path.exists():
        shutil.move(str(female_path), str(TRAIN_PATH / "female"))
    if male_path.exists():
        shutil.move(str(male_path), str(TRAIN_PATH / "male"))
else:
    print("Dataset already available at:", FACES_PATH)

assert TRAIN_PATH.exists(), f"Training path not found: {TRAIN_PATH}"
assert BENCHMARK_PATH.exists(), f"Benchmark path not found: {BENCHMARK_PATH}"

print("Training path:", TRAIN_PATH)
print("Benchmark path:", BENCHMARK_PATH)


## 3. Data transforms

We separate two transform pipelines:
- a **display transform** for visualization
- a **model transform** for training and evaluation

This keeps the notebook easier to understand because normalized tensors are great for the model, but not very nice to display directly.


In [ ]:

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

display_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

if USE_DATA_AUGMENTATION:
    train_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    ])
else:
    train_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    ])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])


## 4. Load datasets

We load the same image files more than once:
- once with the **display** transform
- once with the **training/evaluation** transforms

This is a very practical pattern in workshops because it avoids a common beginner problem: plotting normalized images and wondering why they look strange.


In [ ]:

full_train_display = datasets.ImageFolder(TRAIN_PATH, transform=display_transform)
full_train_model = datasets.ImageFolder(TRAIN_PATH, transform=train_transform)
full_train_eval = datasets.ImageFolder(TRAIN_PATH, transform=eval_transform)
benchmark_display = datasets.ImageFolder(BENCHMARK_PATH, transform=display_transform)
benchmark_model = datasets.ImageFolder(BENCHMARK_PATH, transform=eval_transform)

class_names = full_train_model.classes
num_classes = len(class_names)

assert full_train_display.classes == full_train_model.classes == benchmark_model.classes

n_total = len(full_train_model)
n_val = int(VALIDATION_SPLIT * n_total)
n_train = n_total - n_val

generator = torch.Generator().manual_seed(SEED)

train_model_ds, val_model_ds = random_split(full_train_model, [n_train, n_val], generator=generator)
_, val_eval_ds = random_split(full_train_eval, [n_train, n_val], generator=torch.Generator().manual_seed(SEED))
train_display_ds, val_display_ds = random_split(full_train_display, [n_train, n_val], generator=torch.Generator().manual_seed(SEED))

train_loader = DataLoader(train_model_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_eval_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
benchmark_loader = DataLoader(benchmark_model, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print("Classes:", class_names)
print(f"Train samples:      {len(train_model_ds)}")
print(f"Validation samples: {len(val_eval_ds)}")
print(f"Benchmark samples:  {len(benchmark_model)}")


## 5. Inspect the data

Before training any model, we should first understand the dataset.

### 5.1 Class balance


In [ ]:

from collections import Counter

train_targets = [full_train_model.samples[i][1] for i in range(len(full_train_model))]
benchmark_targets = [benchmark_model.samples[i][1] for i in range(len(benchmark_model))]

train_counts = Counter(train_targets)
benchmark_counts = Counter(benchmark_targets)

summary_df = pd.DataFrame({
    "class": class_names,
    "train_count": [train_counts[i] for i in range(num_classes)],
    "benchmark_count": [benchmark_counts[i] for i in range(num_classes)],
})
summary_df


In [ ]:

summary_df.set_index("class").plot(kind="bar", figsize=(7,4))
plt.title("Class balance")
plt.ylabel("Number of images")
plt.xticks(rotation=0)
plt.show()


### Checkpoint questions
- Are the classes balanced?
- If the classes were strongly imbalanced, would accuracy alone still be enough?
- Which other metric(s) would you want to inspect?


### 5.2 Browse some examples

Use the widget below to look at a few training images.


In [ ]:

import ipywidgets as widgets
from IPython.display import display

def show_example_grid(dataset, class_names, n=12, seed=SEED):
    rng = np.random.default_rng(seed)
    indices = rng.choice(len(dataset), size=min(n, len(dataset)), replace=False)
    cols = 4
    rows = int(np.ceil(len(indices) / cols))
    plt.figure(figsize=(3.5 * cols, 3.5 * rows))
    for i, idx in enumerate(indices, 1):
        img, label = dataset[idx]
        img = img.permute(1, 2, 0).numpy()
        plt.subplot(rows, cols, i)
        plt.imshow(img)
        plt.title(class_names[label])
        plt.axis("off")
    plt.tight_layout()
    plt.show()

show_example_grid(full_train_display, class_names, n=SHOW_DATASET_EXAMPLES)


In [ ]:

def browse_dataset(dataset, class_names):
    def _show(i=0):
        img, label = dataset[i]
        img = img.permute(1, 2, 0).numpy()
        plt.figure(figsize=(4,4))
        plt.imshow(img)
        plt.title(f"Index: {i} | Label: {class_names[label]}")
        plt.axis("off")
        plt.show()

    slider = widgets.IntSlider(
        value=0, min=0, max=len(dataset)-1, step=1, description="sample"
    )
    return widgets.interact(_show, i=slider)

# browse_dataset(full_train_display, class_names)


### Your turn
Look through a few samples and discuss:
- What kinds of variation do you see?
- What image characteristics might make classification difficult?
- What shortcuts might the model learn if we are not careful?


## 6. Sanity checks before training

These checks are simple but extremely valuable. They help you verify that:
- the dataloaders work
- labels are correct
- the model will receive the shape we expect

A lot of training bugs can be caught right here.


In [ ]:

images, labels = next(iter(train_loader))
print("Batch image shape:", images.shape)
print("Batch label shape:", labels.shape)
print("Label ids in batch:", labels[:10].tolist())
print("Class mapping:", full_train_model.class_to_idx)


In [ ]:

# Visualize a batch after undoing normalization
def denormalize(img_tensor, mean=imagenet_mean, std=imagenet_std):
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return img_tensor * std + mean

plt.figure(figsize=(10, 6))
for i in range(min(8, len(images))):
    plt.subplot(2, 4, i + 1)
    img = denormalize(images[i]).permute(1, 2, 0).clamp(0, 1).numpy()
    plt.imshow(img)
    plt.title(class_names[labels[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()


## 7. Build the model

Choose a pretrained backbone by setting `MODEL_NAME` in the control panel:

| `MODEL_NAME` | Architecture | Trainable params (frozen) | Notes |
|---|---|---|---|
| `"mobilenet_v2"` | MobileNetV2 | ~1.2 M | Fast; good default for short sessions |
| `"resnet50"` | ResNet-50 | ~2.1 M | Classic CNN; stronger baseline |
| `"vit_b_16"` | ViT-B/16 | ~0.7 M | Transformer; set `IMG_SIZE = 224` |

All three are pretrained on ImageNet. The original classifier head is replaced
with a dropout + linear layer for our binary task.

In [ ]:
def build_model(model_name=MODEL_NAME, num_classes=2, dropout=0.2,
                freeze_backbone=True, unfreeze_last_block=False):
    if model_name == "mobilenet_v2":
        base = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V2)
        if freeze_backbone:
            for param in base.features.parameters():
                param.requires_grad = False
        if unfreeze_last_block:
            for param in base.features[-1].parameters():
                param.requires_grad = True
        in_features = base.classifier[1].in_features
        base.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, num_classes),
        )

    elif model_name == "resnet50":
        base = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        if freeze_backbone:
            for name, param in base.named_parameters():
                if not name.startswith("fc"):
                    param.requires_grad = False
        if unfreeze_last_block:
            for param in base.layer4.parameters():
                param.requires_grad = True
        in_features = base.fc.in_features
        base.fc = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, num_classes),
        )

    elif model_name == "vit_b_16":
        base = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
        if freeze_backbone:
            for name, param in base.named_parameters():
                if not name.startswith("heads"):
                    param.requires_grad = False
        if unfreeze_last_block:
            for param in base.encoder.layers[-1].parameters():
                param.requires_grad = True
        in_features = base.heads.head.in_features
        base.heads.head = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, num_classes),
        )

    else:
        raise ValueError(
            f"Unknown MODEL_NAME {model_name!r}. "
            "Choose 'mobilenet_v2', 'resnet50', or 'vit_b_16'."
        )

    return base

model = build_model(
    model_name=MODEL_NAME,
    num_classes=num_classes,
    dropout=DROPOUT,
    freeze_backbone=FREEZE_BACKBONE,
    unfreeze_last_block=UNFREEZE_LAST_BLOCK,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Backbone: {MODEL_NAME}")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

### Checkpoint questions
- What does it mean to **freeze** the backbone?
- Why might we prefer fewer trainable parameters in a short workshop?
- What do you predict will happen if we unfreeze more layers?


In [ ]:

with torch.no_grad():
    logits = model(images.to(device))
print("Model output shape:", logits.shape)
print("Expected shape:    ", (images.shape[0], num_classes))


## 8. Training utilities

The next cell defines helper functions for:
- one training epoch
- one evaluation pass
- plotting learning curves
- optional TensorBoard logging

You do not need to edit it during the workshop, but it is worth skimming once so you know where the metrics come from.


In [ ]:
from tqdm.auto import tqdm
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
import seaborn as sns


def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for images, labels in tqdm(dataloader, leave=False, desc="Training"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        running_total += labels.size(0)

    return {
        "loss": running_loss / running_total,
        "accuracy": running_correct / running_total,
    }


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_total = 0
    all_labels = []
    all_preds = []

    for images, labels in tqdm(dataloader, leave=False, desc="Evaluating"):
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)
        preds = logits.argmax(dim=1)

        running_loss += loss.item() * images.size(0)
        running_total += labels.size(0)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

    labels_np = np.array(all_labels)
    preds_np = np.array(all_preds)

    return {
        "loss": running_loss / running_total,
        "accuracy": accuracy_score(labels_np, preds_np),
        "precision": precision_score(labels_np, preds_np, average="weighted", zero_division=0),
        "recall": recall_score(labels_np, preds_np, average="weighted", zero_division=0),
        "f1": f1_score(labels_np, preds_np, average="weighted", zero_division=0),
        "labels": labels_np,
        "preds": preds_np,
    }


def log_metrics_to_tensorboard(writer, metrics, split, epoch):
    for metric_name in ["loss", "accuracy", "precision", "recall", "f1"]:
        if metric_name in metrics:
            writer.add_scalar(f"{split}/{metric_name}", float(metrics[metric_name]), epoch)


def fit(model, train_loader, val_loader, criterion, optimizer, device, epochs, writer=None):
    history = {
        "train_loss": [], "train_acc": [],
        "val_loss": [], "val_acc": [],
        "val_precision": [], "val_recall": [], "val_f1": []
    }

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    if writer is not None:
        writer.add_scalar("model/trainable_parameters", trainable_params, 0)
        writer.add_scalar("model/total_parameters", total_params, 0)

    for epoch in range(1, epochs + 1):
        print(f"Epoch {epoch}/{epochs}")
        train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_metrics = evaluate(model, val_loader, criterion, device)

        history["train_loss"].append(train_metrics["loss"])
        history["train_acc"].append(train_metrics["accuracy"])
        history["val_loss"].append(val_metrics["loss"])
        history["val_acc"].append(val_metrics["accuracy"])
        history["val_precision"].append(val_metrics["precision"])
        history["val_recall"].append(val_metrics["recall"])
        history["val_f1"].append(val_metrics["f1"])

        print(
            f"train_loss={train_metrics['loss']:.4f} | train_acc={train_metrics['accuracy']:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} | val_acc={val_metrics['accuracy']:.4f} | "
            f"val_f1={val_metrics['f1']:.4f}"
        )

        if writer is not None:
            log_metrics_to_tensorboard(writer, train_metrics, "train", epoch)
            log_metrics_to_tensorboard(writer, val_metrics, "validation", epoch)
            writer.add_scalar("optimizer/learning_rate", optimizer.param_groups[0]["lr"], epoch)

    return history


def plot_history(history):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(history["train_loss"], label="train")
    axes[0].plot(history["val_loss"], label="validation")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(history["train_acc"], label="train")
    axes[1].plot(history["val_acc"], label="validation")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    axes[2].plot(history["val_precision"], label="precision")
    axes[2].plot(history["val_recall"], label="recall")
    axes[2].plot(history["val_f1"], label="f1")
    axes[2].set_title("Validation metrics")
    axes[2].set_xlabel("Epoch")
    axes[2].legend()

    plt.tight_layout()
    plt.show()


def plot_confusion(labels, preds, class_names, title="Confusion matrix"):
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.show()


## 9. Train the model

### Expected observation
Because this is transfer learning with a pretrained backbone, the model should usually start learning quite quickly.

As the training runs, watch the loss and accuracy carefully:
- Is the validation accuracy improving?
- Is the validation loss behaving similarly to the training loss?


In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE
)

history = fit(model, train_loader, val_loader, loss_fn, optimizer, device, EPOCHS)


In [ ]:

plot_history(history)


### Reflection prompt
Look at the learning curves above.
- Is the model still improving at the end?
- Do you see signs of overfitting?
- If you had one more epoch budget, would you keep the setup or change something first?


## 10. Evaluate on validation and benchmark sets

Now we compare performance on:
- the **validation** split used during development
- the **benchmark** split kept aside until the end


In [ ]:

val_results = evaluate(model, val_loader, loss_fn, device)
benchmark_results = evaluate(model, benchmark_loader, loss_fn, device)

print(f"Validation loss:     {val_results['loss']:.4f}")
print(f"Validation accuracy: {val_results['accuracy']:.4f}")
print()
print(f"Benchmark loss:      {benchmark_results['loss']:.4f}")
print(f"Benchmark accuracy:  {benchmark_results['accuracy']:.4f}")


In [ ]:
metrics_df = pd.DataFrame([
    {
        "split": "validation",
        "accuracy": val_results["accuracy"],
        "precision": val_results["precision"],
        "recall": val_results["recall"],
        "f1": val_results["f1"],
    },
    {
        "split": "benchmark",
        "accuracy": benchmark_results["accuracy"],
        "precision": benchmark_results["precision"],
        "recall": benchmark_results["recall"],
        "f1": benchmark_results["f1"],
    },
])
metrics_df


In [ ]:

plot_confusion(benchmark_results["labels"], benchmark_results["preds"], class_names, title="Benchmark confusion matrix")


### Checkpoint questions
- Why do we keep a separate benchmark set?
- What would it mean if training accuracy is high but benchmark accuracy is much lower?
- When do precision and recall matter more than plain accuracy?


## 11. Interactive predictions

This section is here to make the workshop more hands-on:
- browse benchmark images
- inspect predicted probabilities
- compare prediction vs. true label

Use it to look for patterns, not just isolated examples.


In [ ]:

@torch.no_grad()
def predict_single_image(model, dataset_display, dataset_model, index, class_names):
    img_display, true_label = dataset_display[index]
    img_model, _ = dataset_model[index]

    logits = model(img_model.unsqueeze(0).to(device))
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred_label = int(np.argmax(probs))

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(img_display.permute(1, 2, 0).numpy())
    axes[0].set_title(
        f"True: {class_names[true_label]}\nPred: {class_names[pred_label]}"
    )
    axes[0].axis("off")

    axes[1].barh(class_names, probs)
    axes[1].set_xlim(0, 1)
    axes[1].set_title("Predicted probabilities")
    axes[1].set_xlabel("Probability")
    plt.tight_layout()
    plt.show()

    return {
        "true_label": class_names[true_label],
        "pred_label": class_names[pred_label],
        "probs": probs
    }

def interactive_prediction_browser():
    def _show(index=0):
        return predict_single_image(model, benchmark_display, benchmark_model, index, class_names)

    slider = widgets.IntSlider(
        value=0, min=0, max=len(benchmark_display)-1, step=1, description="image"
    )
    return widgets.interact(_show, index=slider)

interactive_prediction_browser()


### Your turn
Browse a few examples and look for:
- highly confident correct predictions
- uncertain predictions
- confident mistakes

Which cases are the most interesting to discuss, and why?


## 12. Error analysis

A model becomes much easier to understand when you inspect its failures directly.

The next cells collect benchmark predictions into a table so you can sort by confidence and inspect mistakes.


In [ ]:

def collect_predictions(model, dataloader, device):
    model.eval()
    rows = []
    dataset = dataloader.dataset

    idx_global = 0
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            logits = model(images)
            probs = torch.softmax(logits, dim=1)
            preds = logits.argmax(dim=1).cpu()

            for i in range(len(labels)):
                rows.append({
                    "dataset_index": idx_global,
                    "true": int(labels[i]),
                    "pred": int(preds[i]),
                    "correct": bool(preds[i] == labels[i]),
                    "confidence": float(probs[i, preds[i]].cpu()),
                })
                idx_global += 1
    return pd.DataFrame(rows)

benchmark_pred_df = collect_predictions(model, benchmark_loader, device)
benchmark_pred_df["true_name"] = benchmark_pred_df["true"].map(lambda x: class_names[x])
benchmark_pred_df["pred_name"] = benchmark_pred_df["pred"].map(lambda x: class_names[x])

benchmark_pred_df.head()


In [ ]:

misclassified_df = benchmark_pred_df.query("correct == False").sort_values("confidence", ascending=False)
correct_df = benchmark_pred_df.query("correct == True").sort_values("confidence", ascending=False)

print("Misclassified samples:", len(misclassified_df))
print("Correctly classified samples:", len(correct_df))
misclassified_df.head(10)


In [ ]:

def show_prediction_rows(df, display_dataset, n=6):
    n = min(n, len(df))
    if n == 0:
        print("No rows to show.")
        return

    sample_df = df.head(n)
    plt.figure(figsize=(12, 3*n))
    for plot_i, (_, row) in enumerate(sample_df.iterrows(), 1):
        img, _ = display_dataset[int(row["dataset_index"])]
        img = img.permute(1, 2, 0).numpy()

        plt.subplot(n, 1, plot_i)
        plt.imshow(img)
        plt.title(
            f"true={row['true_name']} | pred={row['pred_name']} | "
            f"confidence={row['confidence']:.3f}"
        )
        plt.axis("off")
    plt.tight_layout()
    plt.show()

show_prediction_rows(misclassified_df, benchmark_display, n=5)


### Reflection prompt
Look at the mistakes above. Which of these explanations seem plausible?
- image quality
- pose or angle
- lighting
- hairstyle or accessories
- background/context cues
- ambiguity in the labels


## 13. Grad-CAM (optional extension)

Grad-CAM helps visualize which regions of an image influence the model's prediction.

The target layer is selected automatically based on `MODEL_NAME`:
- **MobileNetV2**: `model.features[-1]`
- **ResNet-50**: `model.layer4[-1]`
- **ViT-B/16**: `model.encoder.layers[-1].ln_1`

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

if MODEL_NAME == "mobilenet_v2":
    target_layer = model.features[-1]
elif MODEL_NAME == "resnet50":
    target_layer = model.layer4[-1]
elif MODEL_NAME == "vit_b_16":
    target_layer = model.encoder.layers[-1].ln_1
else:
    raise ValueError(f"No Grad-CAM target defined for {MODEL_NAME!r}")

for param in target_layer.parameters():
    param.requires_grad = True

def gradcam_for_index(index, target_class=None):
    img_display, _ = benchmark_display[index]
    img_model, _ = benchmark_dataset[index]

    img_np = np.array(img_display).astype(np.float32) / 255.0
    input_tensor = img_model.unsqueeze(0).to(device)

    with GradCAM(model=model, target_layers=[target_layer]) as cam:
        targets = [ClassifierOutputTarget(target_class)] if target_class is not None else None
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]

    visualization = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
    return visualization

### Final discussion prompts
- Is the model looking at the face region, or something else?
- Do the explanations increase your trust in the model?
- Can Grad-CAM show **where** the model is looking without proving **why** the decision is valid?


## 14. Hyperparameter tuning with TensorBoard (optional extension)

Only after you have completed the main workflow should you start experimenting.

In this section you can run a few small experiments and save them to **TensorBoard**, which is a tool for comparing training curves across runs.

### What we will log
For each run, we will save:
- training loss and accuracy
- validation loss and accuracy
- validation precision, recall, and F1
- learning rate
- total and trainable parameter counts


In [ ]:
from datetime import datetime
from itertools import product
from pathlib import Path

RUNS_DIR = DATA_PATH / "tensorboard_runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print("TensorBoard logs will be written to:", RUNS_DIR)

# Keep this grid small for workshop use.
TUNING_GRID = {
    "model_name": ["mobilenet_v2", "resnet50", "vit_b_16"],
    "lr": [1e-4, 5e-4],
    "dropout": [0.2, 0.5],
    "freeze_backbone": [True],
    "unfreeze_last_block": [False],
}
TUNING_EPOCHS = 3


def iter_experiments(grid):
    keys = list(grid.keys())
    for values in product(*[grid[k] for k in keys]):
        yield dict(zip(keys, values))

### How to launch TensorBoard

After you have run the tuning cell below, use **one** of these options:

#### In Jupyter / local notebook
```python
%load_ext tensorboard
%tensorboard --logdir {RUNS_DIR}
```

#### In a terminal
```bash
tensorboard --logdir path/to/tensorboard_runs
```
Then open the local URL shown by TensorBoard in your browser.

### What to compare in TensorBoard
Open the **Scalars** tab and compare runs using:
- `train/loss` and `validation/loss`
- `train/accuracy` and `validation/accuracy`
- `validation/precision`, `validation/recall`, `validation/f1`
- `optimizer/learning_rate`

Ask yourself:
- Which setting converges fastest?
- Which setting gives the best validation F1?
- Do any runs look unstable or overfit quickly?


In [ ]:
loss_fn = nn.CrossEntropyLoss()

tuning_results = []
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

for run_idx, config in enumerate(iter_experiments(TUNING_GRID), start=1):
    run_name = (
        f"run_{run_idx:02d}_"
        f"{config['model_name']}_"
        f"lr{config['lr']}_"
        f"drop{config['dropout']}"
    )
    print(f"\n=== Starting {run_name} ===")

    model = build_model(
        model_name=config["model_name"],
        num_classes=num_classes,
        dropout=config["dropout"],
        freeze_backbone=config["freeze_backbone"],
        unfreeze_last_block=config["unfreeze_last_block"],
    ).to(device)

    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad],
        lr=config["lr"],
    )

    writer = SummaryWriter(log_dir=str(RUNS_DIR / f"{timestamp}_{run_name}"))
    writer.add_text("config", str(config))

    history = fit(
        model,
        train_loader,
        val_loader,
        loss_fn,
        optimizer,
        device,
        TUNING_EPOCHS,
        writer=writer,
    )

    final_val = evaluate(model, val_loader, loss_fn, device)
    final_benchmark = evaluate(model, benchmark_loader, loss_fn, device)

    for split_name, metrics in [("benchmark", final_benchmark)]:
        for metric_name in ["loss", "accuracy", "precision", "recall", "f1"]:
            writer.add_scalar(f"{split_name}/{metric_name}", float(metrics[metric_name]), TUNING_EPOCHS)

    writer.flush()
    writer.close()

    tuning_results.append({
        "run_name": run_name,
        **config,
        "val_accuracy": final_val["accuracy"],
        "val_f1": final_val["f1"],
        "benchmark_accuracy": final_benchmark["accuracy"],
        "benchmark_f1": final_benchmark["f1"],
    })

tuning_results_df = pd.DataFrame(tuning_results).sort_values(["val_f1", "val_accuracy"], ascending=False)
tuning_results_df

### Final reflection
After comparing the runs, discuss:
- Which hyperparameters seemed to matter most?
- Did the best validation run also look best on the benchmark set?
- Would you trust the conclusion after only a few epochs, or would you want more evidence?


## 15. Suggested workshop wrap-up

By the end of this notebook, you should now be able to explain:
1. the difference between training, validation, and benchmark data  
2. what transfer learning does  
3. why evaluation needs more than one metric  
4. what kinds of mistakes the model makes  
5. what Grad-CAM can and cannot tell us  
6. how to compare several runs with TensorBoard

Nice work — you now have a complete baseline workflow for a small image-classification project.
